# 02 Train 2D CNN v0.4

**Notebook version:** v0.4  
**Category:** training  
**Purpose:** UI-only tmux control panel for launching and monitoring 2D CNN training runs.  
**Inputs:** `./training-data`, `./validation-data`, `./src/train.py`, `./src/v0.2/training_control_panel.py`  
**Expected outputs:** run folders under `./models/<model_directory>/runs/<run_id>/` with `train.log`, `best.pt`, `latest.pt`, and metadata files.  
**Artifact write mode:** canonical artifacts are written by `src/train.py`; this notebook orchestrates launch/session/log controls and run register entries.  
**Decision supported:** `READY_FOR_LAUNCH` vs `PATCH_REQUIRED`


In [1]:
# Repo Setup
from pathlib import Path
import sys

repo_root = Path.cwd().resolve()
if not (repo_root / 'src').exists():
    for parent in repo_root.parents:
        if (parent / 'src').exists() and (parent / 'training-data').exists() and (parent / 'validation-data').exists():
            repo_root = parent
            break
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

helper_root = (repo_root / 'src' / 'v0.2').resolve()
if str(helper_root) not in sys.path:
    sys.path.insert(0, str(helper_root))

print(f'repo_root={repo_root}')
print(f'helper_root={helper_root}')


repo_root=/home/mitch/development/raccoon-ball/rb-training
helper_root=/home/mitch/development/raccoon-ball/rb-training/src/v0.2


In [2]:
# Panel Config
REPO_ROOT = repo_root
PYTHON_EXECUTABLE = sys.executable
TRAINING_MODULE = 'src.train'

TRAINING_DATA_ROOT = REPO_ROOT / 'training-data'
VALIDATION_DATA_ROOT = REPO_ROOT / 'validation-data'
MODELS_ROOT = REPO_ROOT / 'models'

MODEL_SUFFIX_DEFAULT = '2d-cnn'
MODEL_NAME_DEFAULT = 'fast_v0_2'
LOG_FILENAME = 'train.log'
DEFAULT_LOG_TAIL_LINES = 180
DEFAULT_LOG_POLL_SECONDS = 5.0
RESERVED_PARENT_RUN_ID = '[reserved for future implementation]]'

TRAIN_LAUNCH_DEFAULTS = {
    'seed': 42,
    'batch_size': 8,
    'epochs': 32,
    'learning_rate': 1e-3,
    'weight_decay': 1e-5,
    'early_stopping_patience': 3,
    'change_note': 'tmux control-panel launch via notebook v0.4',
}

print(f'python={PYTHON_EXECUTABLE}')
print(f'train_module={TRAINING_MODULE}')
print(f'models_root={MODELS_ROOT}')


python=/home/mitch/development/raccoon-ball/.venv/bin/python
train_module=src.train
models_root=/home/mitch/development/raccoon-ball/rb-training/models


In [3]:
# Helper Imports
import html
import threading

import ipywidgets as widgets
from IPython.display import display

import training_control_panel as control

print(f'loaded_helper={control.__file__}')


loaded_helper=/home/mitch/development/raccoon-ball/rb-training/src/v0.2/training_control_panel.py


In [4]:
# Control Panel
status_area = widgets.HTML(value='')
session_suggestion = widgets.HTML(value='')

model_name_input = widgets.Text(
    value=MODEL_NAME_DEFAULT,
    description='Model Name',
    layout=widgets.Layout(width='320px'),
)
model_directory_input = widgets.Text(
    value=control.suggest_model_directory(MODELS_ROOT, model_suffix=MODEL_SUFFIX_DEFAULT),
    description='Model Dir',
    layout=widgets.Layout(width='420px'),
)
session_name_input = widgets.Text(
    value='',
    description='Session Name',
    placeholder='auto-generated from model name + run id',
    layout=widgets.Layout(width='420px'),
)

seed_input = widgets.IntText(value=TRAIN_LAUNCH_DEFAULTS['seed'], description='Seed')
batch_size_input = widgets.IntText(value=TRAIN_LAUNCH_DEFAULTS['batch_size'], description='Batch Size')
epochs_input = widgets.IntText(value=TRAIN_LAUNCH_DEFAULTS['epochs'], description='Epochs')
learning_rate_input = widgets.FloatText(value=TRAIN_LAUNCH_DEFAULTS['learning_rate'], description='LR')
weight_decay_input = widgets.FloatText(value=TRAIN_LAUNCH_DEFAULTS['weight_decay'], description='Weight Decay')
early_stop_input = widgets.IntText(value=TRAIN_LAUNCH_DEFAULTS['early_stopping_patience'], description='Early Stop')
enable_lr_scheduler_toggle = widgets.Checkbox(
    value=True,
    description='LR Scheduler',
    indent=False,
    layout=widgets.Layout(width='180px'),
)
change_note_input = widgets.Text(
    value=TRAIN_LAUNCH_DEFAULTS['change_note'],
    description='Change Note',
    layout=widgets.Layout(width='760px'),
)
primary_variable_input = widgets.Text(
    value='',
    description='Primary Var',
    placeholder='e.g. bbox line width 3px -> 1px',
    layout=widgets.Layout(width='760px'),
)
notes_input = widgets.Text(
    value='',
    description='Notes',
    layout=widgets.Layout(width='760px'),
)

run_id_preview = widgets.Text(
    value='',
    description='Run ID',
    disabled=True,
    layout=widgets.Layout(width='320px'),
)
run_register_path = widgets.Text(
    value='',
    description='Run Register',
    disabled=True,
    layout=widgets.Layout(width='100%'),
)
derived_log_path = widgets.Text(
    value='',
    description='Log Path',
    disabled=True,
    layout=widgets.Layout(width='100%'),
)

session_select = widgets.Dropdown(
    options=[],
    value=None,
    description='Sessions',
    layout=widgets.Layout(width='420px'),
)

refresh_sessions_button = widgets.Button(description='Refresh Sessions')
launch_button = widgets.Button(description='Launch', button_style='success')
end_session_button = widgets.Button(description='End Session', button_style='danger')

log_tail_lines_input = widgets.IntText(
    value=DEFAULT_LOG_TAIL_LINES,
    description='Tail Lines',
    layout=widgets.Layout(width='220px'),
)
poll_interval_input = widgets.FloatText(
    value=DEFAULT_LOG_POLL_SECONDS,
    description='Poll Secs',
    layout=widgets.Layout(width='220px'),
)
auto_refresh_toggle = widgets.ToggleButton(
    value=False,
    description='Auto Refresh',
    icon='refresh',
    layout=widgets.Layout(width='180px'),
)
refresh_log_button = widgets.Button(description='Refresh Log', button_style='info')

log_output = widgets.Textarea(
    value='',
    description='',
    layout=widgets.Layout(width='100%', height='420px'),
    disabled=True,
)

_auto_refresh_state = {'thread': None, 'stop_event': None}
_session_name_state = {'last_auto': ''}

def _set_status(message: str, *, is_error: bool = False) -> None:
    color = '#9b1c1c' if is_error else '#0f5132'
    lr_scheduler_text = 'enabled' if bool(enable_lr_scheduler_toggle.value) else 'disabled'
    status_area.value = (
        f"<div style='border:1px solid #d0d7de;padding:8px;border-radius:6px;'>"
        f"<strong style='color:{color};'>Status</strong>"
        f"<div><code>lr_scheduler={lr_scheduler_text}</code></div>"
        f"<div>{html.escape(message)}</div>"
        "</div>"
    )

def _sync_default_session_name(suggested_session: str) -> None:
    current = session_name_input.value.strip()
    last_auto = _session_name_state['last_auto']
    if (not current) or (current == last_auto):
        session_name_input.value = suggested_session
    _session_name_state['last_auto'] = suggested_session

def _update_preview_paths(show_error: bool = False):
    model_name = model_name_input.value.strip()
    model_directory = model_directory_input.value.strip()

    if not model_directory:
        run_id_preview.value = ''
        run_register_path.value = ''
        derived_log_path.value = ''
        session_suggestion.value = '<em>Suggested session name: <code>rb-&lt;model_name&gt;-&lt;run_id&gt;</code></em>'
        return None

    try:
        next_run_id = control.preview_next_run_id(
            models_root=MODELS_ROOT,
            model_directory=model_directory,
        )
        run_id_preview.value = next_run_id

        model_dir = control.build_model_dir(
            models_root=MODELS_ROOT,
            model_directory=model_directory,
        )
        register_path = model_dir / 'run_register.json'
        run_register_path.value = str(register_path)

        log_path = control.build_log_path(
            run_id=next_run_id,
            models_root=MODELS_ROOT,
            model_directory=model_directory,
            log_filename=LOG_FILENAME,
        )
        derived_log_path.value = str(log_path)

        session_base = model_name if model_name else 'model'
        suggested_session = f'rb-{session_base}-{next_run_id}'
        session_suggestion.value = f"<em>Suggested session name: <code>{html.escape(suggested_session)}</code></em>"
        _sync_default_session_name(suggested_session)

        return {
            'model_name': model_name,
            'model_directory': model_directory,
            'run_id': next_run_id,
            'run_register_path': str(register_path),
            'log_path': str(log_path),
            'suggested_session': suggested_session,
        }
    except Exception as exc:
        run_id_preview.value = ''
        run_register_path.value = ''
        derived_log_path.value = ''
        session_suggestion.value = '<em>Suggested session name: <code>rb-&lt;model_name&gt;-&lt;run_id&gt;</code></em>'
        if show_error:
            _set_status(f'Path derivation failed: {exc}', is_error=True)
        return None

def _refresh_sessions(*, preferred: str | None = None) -> None:
    try:
        sessions = control.list_sessions()
    except Exception as exc:
        session_select.options = []
        session_select.value = None
        _set_status(f'Session refresh failed: {exc}', is_error=True)
        return

    current = preferred if preferred is not None else session_select.value
    session_select.options = sessions
    if sessions:
        if current in sessions:
            session_select.value = current
        else:
            session_select.value = sessions[0]
    else:
        session_select.value = None

def _resolve_log_path_from_selected_session() -> str | None:
    selected = session_select.value or session_name_input.value.strip()
    if not selected:
        preview = _update_preview_paths(show_error=False)
        return None if preview is None else preview['log_path']

    try:
        resolved = control.resolve_session_run(
            session_name=selected,
            models_root=MODELS_ROOT,
        )
    except Exception as exc:
        _set_status(f'Session lookup failed: {exc}', is_error=True)
        return None

    if resolved is None:
        return None

    model_directory_input.value = resolved['model_directory']
    run_id_preview.value = resolved['run_id']
    derived_log_path.value = resolved['log_path']
    return str(resolved['log_path'])

def _refresh_log(*_) -> None:
    selected_session = session_select.value or session_name_input.value.strip()
    log_path = _resolve_log_path_from_selected_session()
    if log_path is None:
        if selected_session:
            log_output.value = f'[no registered run for session: {selected_session}]'
        else:
            log_output.value = '[no session selected]'
        return

    try:
        tail_lines = int(log_tail_lines_input.value)
        log_output.value = control.read_log_tail(log_path, max_lines_or_chars=tail_lines)
    except Exception as exc:
        log_output.value = ''
        _set_status(f'Log refresh failed: {exc}', is_error=True)

def _stop_auto_refresh() -> None:
    stop_event = _auto_refresh_state.get('stop_event')
    thread = _auto_refresh_state.get('thread')
    if stop_event is not None:
        stop_event.set()
    if thread is not None and thread.is_alive():
        thread.join(timeout=1.0)
    _auto_refresh_state['thread'] = None
    _auto_refresh_state['stop_event'] = None

def _auto_refresh_loop(stop_event: threading.Event) -> None:
    while not stop_event.is_set():
        try:
            _refresh_log()
        except Exception as exc:
            _set_status(f'Auto refresh failed: {exc}', is_error=True)
        interval = max(0.5, float(poll_interval_input.value))
        stop_event.wait(interval)

def _on_auto_refresh_toggle(change):
    enabled = bool(change['new'])
    if enabled:
        _stop_auto_refresh()
        stop_event = threading.Event()
        thread = threading.Thread(target=_auto_refresh_loop, args=(stop_event,), daemon=True)
        _auto_refresh_state['thread'] = thread
        _auto_refresh_state['stop_event'] = stop_event
        thread.start()
        _set_status('Auto refresh enabled.')
    else:
        _stop_auto_refresh()
        _set_status('Auto refresh disabled.')

def _on_launch_clicked(_):
    preview = _update_preview_paths(show_error=True)
    if preview is None:
        return

    session_name = session_name_input.value.strip()
    if not session_name:
        _set_status(
            f"Session name is required. Suggested: {preview['suggested_session']}",
            is_error=True,
        )
        return

    if control.session_exists(session_name):
        _set_status(f'Duplicate session name refused: {session_name}', is_error=True)
        return

    try:
        reservation = control.reserve_run(
            models_root=MODELS_ROOT,
            model_directory=preview['model_directory'],
            model_name=preview['model_name'],
            session_name=session_name,
            primary_variable_changed=primary_variable_input.value.strip(),
            notes=notes_input.value.strip(),
            parent_run_id=RESERVED_PARENT_RUN_ID,
        )

        run_id_preview.value = reservation['run_id']
        run_register_path.value = reservation['run_register_path']
        derived_log_path.value = reservation['log_path']

        command = control.build_launch_command(
            run_id=reservation['run_id'],
            model_directory=preview['model_directory'],
            model_architecture_variant=preview['model_name'],
            python_executable=PYTHON_EXECUTABLE,
            training_module=TRAINING_MODULE,
            training_data_root=TRAINING_DATA_ROOT,
            validation_data_root=VALIDATION_DATA_ROOT,
            output_root=MODELS_ROOT,
            seed=int(seed_input.value),
            batch_size=int(batch_size_input.value),
            epochs=int(epochs_input.value),
            learning_rate=float(learning_rate_input.value),
            weight_decay=float(weight_decay_input.value),
            early_stopping_patience=int(early_stop_input.value),
            enable_lr_scheduler=bool(enable_lr_scheduler_toggle.value),
            change_note=change_note_input.value.strip() or 'tmux control-panel launch',
        )

        control.launch_session(
            session_name=session_name,
            command=command,
            log_path=reservation['log_path'],
            working_directory=REPO_ROOT,
        )

        _refresh_sessions(preferred=session_name)
        _refresh_log()
        _set_status(
            f"Launched session {session_name} for {preview['model_directory']} / {reservation['run_id']}"
        )
    except Exception as exc:
        _set_status(f'Launch failed: {exc}', is_error=True)

def _on_end_session_clicked(_):
    selected_session = session_select.value
    if not selected_session:
        _set_status('Select a session to end.', is_error=True)
        return
    try:
        ended = control.end_session(selected_session)
        if ended:
            _set_status(f'Ended session {selected_session}.')
        else:
            _set_status(f'Session not found: {selected_session}', is_error=True)
        _refresh_sessions()
        _refresh_log()
    except Exception as exc:
        _set_status(f'End session failed: {exc}', is_error=True)

def _on_session_changed(change):
    selected = change['new']
    if selected:
        session_name_input.value = selected
    _refresh_log()

def _on_preview_inputs_changed(_):
    _update_preview_paths(show_error=False)

launch_button.on_click(_on_launch_clicked)
end_session_button.on_click(_on_end_session_clicked)
refresh_sessions_button.on_click(lambda _: _refresh_sessions())
refresh_log_button.on_click(_refresh_log)
session_select.observe(_on_session_changed, names='value')
model_name_input.observe(_on_preview_inputs_changed, names='value')
model_directory_input.observe(_on_preview_inputs_changed, names='value')
auto_refresh_toggle.observe(_on_auto_refresh_toggle, names='value')

_update_preview_paths(show_error=False)
_refresh_sessions()
_set_status('Control panel ready.')

display(widgets.VBox([
    status_area,
    widgets.HBox([model_name_input, model_directory_input]),
    session_name_input,
    session_suggestion,
    widgets.HBox([seed_input, batch_size_input, epochs_input]),
    widgets.HBox([learning_rate_input, weight_decay_input, early_stop_input, enable_lr_scheduler_toggle]),
    change_note_input,
    primary_variable_input,
    notes_input,
    widgets.HBox([run_id_preview]),
    run_register_path,
    derived_log_path,
    widgets.HBox([launch_button, refresh_sessions_button, end_session_button]),
    session_select,
    widgets.HBox([refresh_log_button, log_tail_lines_input, poll_interval_input, auto_refresh_toggle]),
    log_output,
]))


## Final Verdict
`READY_FOR_LAUNCH` when the panel derives `model_directory + run_id` paths, writes `run_register.json`, launches tmux, and refreshes that run's log tail.
